<a href="https://colab.research.google.com/github/mscampb4-ncsu/Homework7/blob/main/Homework7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 7 - ST 554
Author: Max Campbell

## Part 1: Modeling Wine Alcohol Content

In this assignment, we are interested in modeling some of the characteristics present in the [Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality) dataset provided by the UC Irvine Machine Learning Repository. The data itself is sourced from a sampling of various red and white wines from the north of Portugal. We are focused on two main tasks in the modeling process. The first is trying to regress upon the `alcohol` variable to see how accurately we can predict alcohol content, given some other characteristics of the wine sample. The second is trying to classify a wine as white or red depending on those characteristics. We will start with the former of the two tasks in this part.

As with any dataset, we need to read it in:

In [12]:
#Import necessary packages
import pandas as pd

#Read in the winequality datasets.
winequalityRed = pd.read_csv("winequality-red.csv", sep = ";")
winequalityRed['color'] = 'red'
winequalityWhite = pd.read_csv("winequality-white.csv", sep = ";")
winequalityWhite['color'] = 'white'

wine = pd.concat([winequalityRed, winequalityWhite], ignore_index = True)
wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,color
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


Note that in the process of reading in the data, the `color` variable was added to represent which dataset the observation came from. This wasn't included initially as the two datasets were seperated by wine color, so there was no need to have a variable in that format that explained the wine color as it would've been redundant.

Now, we can get to the fun part: modeling!

First we will split up the data into a "training set" and a "test set". We fit the models themselves on the training set, which will generally be the majority of the data, and then evaluate the model's performance on data it hasn't seen before in the test set. We will use functionality provided by the `sklearn` module to do this efficiently:

In [14]:
#Import necessary packages for modeling data
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, LassoCV, Lasso

#Perform a Train/Test split.

X_train, X_test, y_train, y_test = train_test_split(
    wine.drop('alcohol', axis = 1),
    wine['alcohol'],
    test_size = 0.20,
    random_state = 329
)
X_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,color
2954,7.6,0.19,0.41,1.10,0.040,38.0,143.0,0.99070,2.92,0.42,5,white
5874,5.7,0.22,0.20,16.00,0.044,41.0,113.0,0.99862,3.22,0.46,6,white
2849,5.3,0.26,0.23,5.15,0.034,48.0,160.0,0.99520,3.82,0.51,7,white
371,7.9,0.24,0.40,1.60,0.056,11.0,25.0,0.99670,3.32,0.87,6,red
2762,5.7,0.26,0.27,4.10,0.201,73.5,189.5,0.99420,3.27,0.38,6,white


### Model Training

